In [2]:
!pip install nltk scikit-learn pandas numpy

In [3]:
import json
import random
import pickle
import webbrowser

import pandas as pd
import numpy as np

from datetime import datetime

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

In [4]:
from google.colab import files

uploaded = files.upload()

Saving intents.json to intents.json


In [5]:
with open("intents.json", "r") as file:
    data = json.load(file)

print("Dataset loaded successfully!")
print("Total Intents:", len(data["intents"]))

Dataset loaded successfully!
Total Intents: 16


In [6]:
patterns = []
tags = []

for intent in data["intents"]:
    for pattern in intent["patterns"]:
        patterns.append(pattern)
        tags.append(intent["tag"])

df = pd.DataFrame({
    "Pattern": patterns,
    "Tag": tags
})

print(df.head())

          Pattern       Tag
0              Hi  greeting
1           Hello  greeting
2             Hey  greeting
3    Good morning  greeting
4  Good afternoon  greeting


In [7]:
print("Total Training Samples:", len(df))
print()

print(df["Tag"].value_counts())

Total Training Samples: 380

Tag
greeting          25
goodbye           25
thanks            25
identity          25
help              25
date              25
time              25
weather           25
calculator        25
open_website      25
reminder          25
faq               25
jokes             25
motivation        25
unknown           25
show_reminders     5
Name: count, dtype: int64


In [8]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(df["Pattern"])

y = df["Tag"]

print("Feature Matrix Shape:", X.shape)

Feature Matrix Shape: (380, 352)


In [9]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

print("Training Samples:", X_train.shape[0])
print("Testing Samples:", X_test.shape[0])

Training Samples: 304
Testing Samples: 76


In [10]:
model = LogisticRegression(max_iter=1000)

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


In [11]:
predictions = model.predict(X_test)

accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", round(accuracy * 100, 2), "%")

Accuracy: 68.42 %


In [12]:
pickle.dump(model, open("model.pkl", "wb"))
pickle.dump(vectorizer, open("vectorizer.pkl", "wb"))

print("Model and Vectorizer saved successfully!")

Model and Vectorizer saved successfully!


In [13]:
# Create a dictionary to store responses for each intent
response_dict = {}

for intent in data["intents"]:
    response_dict[intent["tag"]] = intent["responses"]

print("Response Dictionary Created Successfully!")

Response Dictionary Created Successfully!


In [14]:
from datetime import datetime

print(datetime.now())

2026-08-17 13:58:50.895112


In [15]:
from datetime import datetime
from zoneinfo import ZoneInfo

# Function to get current date
def get_date():
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%d-%m-%Y")

# Function to get current time
def get_time():
    return datetime.now(ZoneInfo("Asia/Kolkata")).strftime("%I:%M %p")

In [16]:
import re

# Calculator function
def calculate(expression):
    try:
        # Keep only numbers, operators, brackets, and decimal points
        expression = re.sub(r'[^0-9+\-*/(). ]', '', expression)

        result = eval(expression)

        return result

    except Exception:
        return "Sorry! I couldn't calculate that."

In [17]:
import webbrowser

def open_website(user_input):
    user_input = user_input.lower()

    websites = {
        "google": "https://www.google.com",
        "youtube": "https://www.youtube.com",
        "github": "https://github.com",
        "gmail": "https://mail.google.com",
        "wikipedia": "https://www.wikipedia.org",
        "linkedin": "https://www.linkedin.com",
        "chatgpt": "https://chat.openai.com",
        "facebook": "https://www.facebook.com",
        "instagram": "https://www.instagram.com"
    }

    for site in websites:
        if site in user_input:
            webbrowser.open(websites[site])
            return f"Opening {site.title()}..."

    return "Sorry! I couldn't identify the website."

In [18]:
import re

def calculate(user_input):
    try:
        expression = re.findall(r"[0-9+\-*/().]+", user_input)

        if expression:
            expression = "".join(expression)
            result = eval(expression)
            return result
        else:
            return "No mathematical expression found."

    except Exception:
        return "Invalid expression."

In [19]:
!pip install requests

In [20]:
import requests

In [21]:
API_KEY = "a3ddb3696491970152d79e5f22ce1636"

def get_weather(city):
    url = f"https://api.openweathermap.org/data/2.5/weather?q={city}&appid={API_KEY}&units=metric"

    response = requests.get(url)

    if response.status_code == 200:
        data = response.json()

        temperature = data["main"]["temp"]
        humidity = data["main"]["humidity"]
        condition = data["weather"][0]["description"]

        return (
            f"Weather in {city.title()}:\n"
            f"🌡️ Temperature: {temperature}°C\n"
            f"☁️ Condition: {condition.title()}\n"
            f"💧 Humidity: {humidity}%"
        )

    else:
        return "Sorry! I couldn't find that city."

In [22]:
# Dictionary to store reminders
reminders = []

def add_reminder(reminder):
    reminders.append(reminder)
    return "✅ Reminder saved successfully!"

def show_reminders():
    if len(reminders) == 0:
        return "📌 No reminders found."

    result = "📋 Your Reminders:\n"

    for i, reminder in enumerate(reminders, start=1):
        result += f"{i}. {reminder}\n"

    return result

In [23]:
knowledge = {
    "python": "Python is a high-level programming language used for web development, Artificial Intelligence, Machine Learning, data science, automation, and software development.",

    "machine learning": "Machine Learning is a branch of Artificial Intelligence that enables computers to learn from data and make predictions without being explicitly programmed.",

    "artificial intelligence": "Artificial Intelligence is the simulation of human intelligence by machines to perform tasks like learning, reasoning, and decision-making.",

    "apj abdul kalam": "Dr. A. P. J. Abdul Kalam was an Indian aerospace scientist and the 11th President of India. He is widely known as the Missile Man of India.",

    "java": "Java is a popular object-oriented programming language used for web, desktop, mobile, and enterprise application development.",

    "c++": "C++ is a powerful object-oriented programming language widely used in system programming, game development, and competitive programming."
}

In [24]:
def search_knowledge(query):
    query = query.lower().strip()

    if query in knowledge:
        return knowledge[query]
    else:
        return "Sorry! I don't have information about that topic."

In [25]:
def save_chat(user_message, assistant_message):
    with open("chat_history.txt", "a", encoding="utf-8") as file:
        file.write(f"You: {user_message}\n")
        file.write(f"Assistant: {assistant_message}\n")
        file.write("-" * 50 + "\n")

In [26]:
# Final Chatbot -- Here you can ask anything

print("="*60)
print("🤖 AI Virtual Assistant")
print("="*60)
print("I can help you with:")
print("• Greetings")
print("• Date & Time")
print("• Weather Report")
print("• Calculator")
print("• Open Websites")
print("• Motivation Quotes")
print("• Jokes")
print("• Knowledge Base")
print("• Reminders")
print("• Any Help")
print("Type 'exit' to stop.")
print("="*60)

while True:

    user_input = input("\nYou : ")

    if user_input.lower() in ["exit","quit","bye","goodbye"]:
        print("🤖 Assistant : Goodbye! Have a nice day 😊")
        break

     # Show reminders directly
    if user_input.lower() in [
        "show reminders",
        "my reminders",
        "display reminders",
        "view reminders",
        "list reminders"
    ]:
        print("🤖 Assistant :")
        response = show_reminders()
        print("🤖 Assistant :")
        print(response)
        save_chat(user_input, response)
        continue

     # Knowledge Search
    if (
        user_input.lower().startswith("tell me about") or
        user_input.lower().startswith("who is") or
        user_input.lower().startswith("what is")
    ):

        query = user_input.lower()

        for word in ["tell me about", "who is", "what is"]:
            query = query.replace(word, "")

        query = query.strip()

        response = search_knowledge(query)

        print("🤖 Assistant :")
        print(response)

        continue

    # Machine Learning prediction starts here
    vector = vectorizer.transform([user_input])
    predicted_intent = model.predict(vector)[0]

    # Date
    if predicted_intent == "date":
      response = "Today's date is " + get_date()
      print("🤖 Assistant :", response)
      save_chat(user_input, response)

    # Time
    elif predicted_intent == "time":
        response = "Current time is " + get_time()
        print("🤖 Assistant :", response)
        save_chat(user_input, response)

    # Calculator
    elif predicted_intent == "calculator":
        result = calculate(user_input)
        result = calculate(user_input)
        response = "The result is " + str(result)
        print("🤖 Assistant :", response)
        save_chat(user_input, response)

    # Open Website
    elif predicted_intent == "open_website":
        response = open_website(user_input)
        print("🤖 Assistant :", response)
        save_chat(user_input, response)

    # Open weather
    elif predicted_intent == "weather":

        words = user_input.split()

        if len(words) > 1:
            city = " ".join(words[1:])
            response = get_weather(city)
            print("🤖 Assistant :")
            print(response)
            save_chat(user_input, response)
        else:
            print("🤖 Assistant : Please type like this:")
            print("weather Hyderabad")

    #Remainder
    elif predicted_intent == "reminder":
        reminder = input("🤖 Assistant : What should I remind you about?\nYou : ")
        response = add_reminder(reminder)
        print("🤖 Assistant :", response)
        save_chat(user_input, response)

    elif predicted_intent == "show_reminders":
         response = show_reminders()
         print("🤖 Assistant :")
         print(response)
         save_chat(user_input, response)


    # Other Intents
    else:
        response = random.choice(response_dict[predicted_intent])
        print("\n🤖 Assistant :", response)
        save_chat(user_input, response)

🤖 AI Virtual Assistant
I can help you with:
• Greetings
• Date & Time
• Weather Report
• Calculator
• Open Websites
• Motivation Quotes
• Jokes
• Knowledge Base
• Reminders
• Any Help
Type 'exit' to stop.

You : hello

🤖 Assistant : Hi there! How may I assist you?

You : motivate me

🤖 Assistant : Success comes to those who stay consistent. Keep going!

You : current date
🤖 Assistant : Today's date is 17-08-2026

You : current time
🤖 Assistant : Current time is 07:29 PM

You : weather Rajahmundry
🤖 Assistant :
Weather in Rajahmundry:
🌡️ Temperature: 30.13°C
☁️ Condition: Overcast Clouds
💧 Humidity: 79%

You : remind me
🤖 Assistant : What should I remind you about?
You : Application to be filled by tomorrow morning
🤖 Assistant : ✅ Reminder saved successfully!

You : show reminders
🤖 Assistant :
🤖 Assistant :
📋 Your Reminders:
1. Application to be filled by tomorrow morning


You : calculate 2443*43
🤖 Assistant : The result is 105049

You : calculate 564/34
🤖 Assistant : The result is 16